<a href="https://colab.research.google.com/github/smitasasindran/era4/blob/session12/Session12/Era4_Session12_MiniGPT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Import required libraries

In [ ]:
import os
import math
import time
import inspect
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
from torchsummary import summary
import tiktoken


### Set the device and seed

In [ ]:
from model import GPT, GPTConfig, DataLoaderLite

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
print(f"using device: {device}")

# SEED
torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1337)


BATCH_SIZE = 8
TOKEN_SIZE = 512 #256
EPOCHS = 70

using device: cuda


### Create the model from GPT config

In [ ]:
model = GPT(GPTConfig())
model.to(device)

train_loader = DataLoaderLite(B = BATCH_SIZE, T = TOKEN_SIZE)

loaded 338025 tokens
1 epoch = 82 batches


### Optimizer and LR Scheduler

In [ ]:
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingLR, SequentialLR, LinearLR

lr = 0.0005 #3e-4
warmup_epochs = 30
# optimizer = torch.optim.AdamW(model.parameters(), lr = lr)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4, betas=(0.9, 0.999))

start_factor=0.01
total_steps = EPOCHS
warmup_steps = warmup_epochs
tmax = total_steps - warmup_steps
eta_min = 1e-6

warmup = LinearLR(
        optimizer,
        start_factor=start_factor, # start at 1% of base LR
        total_iters=warmup_steps # number of scheduler.step() calls during warmup
    )
cosine = CosineAnnealingLR(
    optimizer,
    T_max=tmax, # number of remaining updates
    eta_min=eta_min
)
scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup, cosine],
    milestones=[warmup_steps] # switch to cosine after warmup
)


### Training Loop

In [ ]:
# While Iterating through epoch steps, the scheduler steps were causing very slow training
# because it was changing LR at every step. Made artificial epochs for this continuous dataloader

for i in range(EPOCHS):
    print(f"\n\nEpoch: {i}")
    b = 0
    while True:
        b += 1
        x, y = train_loader.next_batch()
        # If batches are exhausted
        if train_loader.current_position == 0:
            break

        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits, loss = model(x, y)
        loss.backward()
        optimizer.step()
        print(f'step{b}, loss: {loss.item()}')

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    print(f'Epoch{i}, loss: {loss.item()}, lr: {current_lr}')



print(loss)

Streaming output truncated to the last 5000 lines.
step14, loss: 3.356797695159912
step15, loss: 3.3504271507263184
step16, loss: 3.4538087844848633
step17, loss: 3.13360595703125
step18, loss: 3.2026443481445312
step19, loss: 3.5184295177459717
step20, loss: 3.209160566329956
step21, loss: 3.3301587104797363
step22, loss: 3.0360512733459473
step23, loss: 3.5240628719329834
step24, loss: 3.5200095176696777
step25, loss: 3.5770158767700195
step26, loss: 3.552659034729004
step27, loss: 3.406534194946289
step28, loss: 3.50522518157959
step29, loss: 3.4220118522644043
step30, loss: 3.408039093017578
step31, loss: 3.2617735862731934
step32, loss: 3.1659767627716064
step33, loss: 3.3728363513946533
step34, loss: 3.4891066551208496
step35, loss: 3.5326943397521973
step36, loss: 3.404491901397705
step37, loss: 3.2843854427337646
step38, loss: 3.414073944091797
step39, loss: 3.301892042160034
step40, loss: 3.1719608306884766
step41, loss: 3.260636329650879
step42, loss: 3.444871425628662
step43

### Save the model

In [ ]:
save_file = "gpt2-shakespeare.pth"
checkpoint = {
        "epoch": EPOCHS,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        # "best_loss": best_loss,
        # Add any other relevant information like hyperparameters
    }
torch.save(checkpoint, save_file)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/gpt2-shakespeare.pth /content/drive/MyDrive/era4-data/gpt2-shakespeare-S12.pth

#/content/gpt2-shakespeare.pth

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
gpt2-shakespeare-S12.pth


In [ ]:
!ls /content/drive/MyDrive/era4-data


gpt2-shakespeare-S12.pth


### Testing model

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# STOP
num_return_sequences = 5
max_length = 30


while x.size(1) < max_length:
    # forward the model to get the logits
    with torch.no_grad():
        logits = model(x)[0] # (B, T, vocab_size)
        # take the logits at the last position
        logits = logits[:, -1, :] # (B, vocab_size)
        # get the probabilities
        probs = F.softmax(logits, dim=-1)
        # do top-k sampling of 50 (huggingface pipeline default)
        # topk_probs here becomes (5, 50), topk_indices is (5, 50)
        topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
        # select a token from the top-k probabilities
        # note: multinomial does not demand the input to sum to 1
        ix = torch.multinomial(topk_probs, 1) # (B, 1)
        # gather the corresponding indices
        xcol = torch.gather(topk_indices, -1, ix) # (B, 1)
        # append to the sequence
        x = torch.cat((x, xcol), dim=1)

In [ ]:
# print the generated text
enc = tiktoken.get_encoding('gpt2')
for i in range(num_return_sequences):
    tokens = x[i, :max_length].tolist()
    decoded = enc.decode(tokens)
    print(">", decoded)

> t thyself, wast then her servant;
And, for thou wast a spirit too delicate
To act her earthy and abhorr'd commands
> Shake it off. Come on;
We'll visit Caliban my slave, who never
Yields us kind answer.

M
>  was mine own king: and here you sty me
In this hard rock, whiles you do keep from me
The rest o' the island
>  featly here and there;
And, sweet sprites, the burthen bear.
Hark, hark!

FERDIN
> !
If you be maid or no?

MIRANDA:
No wonder, sir;
But certainly a maid.


